Install dependencies and verify files are present. This assumes a Python environment with torch and torchvision available.</VSCode.Cell>

<VSCode.Cell language="python"># Install tips (run once in your environment, adapt as needed)
# !pip install torch torchvision pandas pillow

import os
print('Working directory:', os.getcwd())


<VSCode.Cell language="python">from model_enhanced import MultiTaskWaterNet
import torch

model = MultiTaskWaterNet(backbone_name='resnet18', pretrained=False, n_classes=4)
model.eval()

x = torch.randn(2, 3, 224, 224)
out = model(x)
print({k: v.shape for k, v in out.items()})


<VSCode.Cell language="python">from dataset_enhanced import WaterDatasetMultiTask

# Create a small CSV `examples/sample.csv` with columns: filename,turbidity_NTU,discolor_label
# and place some images in examples/images/ to try this snippet.

# ds = WaterDatasetMultiTask('examples/sample.csv', 'examples/images', augment=True)
# img, turb, disc, stats = ds[0]
# print(img.shape, turb, disc, stats.shape)
print('Place a small sample csv and images to run dataset demo')

Use `train_enhanced.py` from the command line to train. Example:

```bash
python train_enhanced.py --train_csv data/train.csv --val_csv data/val.csv --img_dir data/images --epochs 20 --batch_size 16
```

Load the saved checkpoint and run a prediction on a single image.</VSCode.Cell>

<VSCode.Cell language="python">import torch
from PIL import Image
from torchvision import transforms
from model_enhanced import MultiTaskWaterNet

# load model
device = torch.device('cpu')
model = MultiTaskWaterNet(backbone_name='resnet18', pretrained=False, n_classes=4)
ckpt_path = 'best_model.pth'
if os.path.exists(ckpt_path):
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state['model_state_dict'])
model.to(device).eval()

# preprocess
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

# replace with your image path
img_p = 'examples/images/example.jpg'
if os.path.exists(img_p):
    img = Image.open(img_p).convert('RGB')
    t = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(t)
    print('Turbidity:', out['turbidity'].cpu().numpy(), 'LogVar:', out['turbidity_logvar'].cpu().numpy())
    print('Discolor logits:', out['discolor_logits'].cpu().numpy())
else:
    print('Put an example image at', img_p)